In [1]:
%load_ext autoreload
%autoreload 2

**Author:** Salvador Navas  
**Date:** 2025-06-27

In [2]:
# General stochastic generation (CoSMoS_py) — discharge, temperature, ...
from pyhydra.climate.stochastic_generation import (
    analyze_ts,
    report_ts,
    simulate_ts,
    generate_ts,
    fit_distribution,
    fit_acs,
)

# Precipitation stochastic generation (NEOPRENE NSRP)
from pyhydra.climate.stochastic_generation import NSRPModel, STNSRPModel

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Stochastic Time Series Generation

This notebook demonstrates two complementary approaches to stochastic time series generation in `pyhydra`:

| Approach | Model | Variables | Library |
|----------|-------|-----------|---------|
| **General** | Seasonal stochastic (CoSMoS) | Any hydro-meteorological variable | CoSMoS_py |
| **Precipitation** | Neyman-Scott Rectangular Pulses (NSRP) | Precipitation only | NEOPRENE |

---

### What is stochastic generation?

A **stochastic generator** produces synthetic time series that are statistically equivalent to an observed record. They are not forecasts — they preserve the statistical structure (distribution, autocorrelation, seasonality) but generate new random sequences. Key applications:

- **Uncertainty quantification** — generate ensembles to propagate input uncertainty through hydrological models
- **Record extension** — produce long synthetic records from short observed ones (e.g. 30 observed years → 1000 synthetic years)
- **Climate stress testing** — perturb observed statistics to represent future or hypothetical conditions

---

### Installation

```bash
# CoSMoS_py (required for Section 1)
pip install -e /path/to/CoSMoS_py
# Source: https://github.com/navass11/CoSMoS_py

# NEOPRENE (optional — required for Sections 2 & 3)
pip install NEOPRENE
# Source: https://github.com/IHCantabria/NEOPRENE
```

---
## 1. 🔵 General Stochastic Generation — CoSMoS_py

CoSMoS fits a **seasonal stochastic model** to any hydro-meteorological variable and generates synthetic realisations that match the observed:
- Seasonal mean and standard deviation
- Marginal probability distribution
- Autocorrelation structure (ACS)

### 💡 Typical use case
Generate an ensemble of synthetic discharge series for uncertainty quantification in hydrological impact studies.

In [3]:
# Generate a synthetic 30-year daily discharge series (observed)
rng   = np.random.default_rng(42)
dates = pd.date_range("1990-01-01", "2019-12-31", freq="D")
n     = len(dates)
doy   = np.asarray(dates.dayofyear, dtype=float)

seasonal_mean = 30 + 20 * np.sin(2 * np.pi * (doy / 365 - 0.3))
Q = np.maximum(0.1, seasonal_mean + rng.normal(0, 8, n))
Q_obs = pd.Series(Q, index=dates, name="discharge")

print(f"Period   : {dates[0].date()} → {dates[-1].date()}")
print(f"Mean Q   : {Q_obs.mean():.2f} m³/s")
print(f"Std  Q   : {Q_obs.std():.2f} m³/s")

Period   : 1990-01-01 → 2019-12-31
Mean Q   : 29.97 m³/s
Std  Q   : 16.11 m³/s


In [4]:
# Fit a marginal distribution to the observed discharge series
dist = fit_distribution(Q_obs, dist='gengamma')
print(f"Fitted distribution : {dist['dist']}")
print(f"Parameters          : {dist['params_dict']}")
print(f"Fit objective (RMSE): {dist['objective']:.4f}")

# Fit the seasonal autocorrelation structure (ACS)
acs = fit_acs(Q_obs, lag_max=30)   # returns list of 12 monthly dicts
print(f"\nACS fitted for {len(acs)} months")
print(f"January  — model: {acs[0]['acs_id']}, params: {acs[0]['params']}")
print(f"July     — model: {acs[6]['acs_id']}, params: {acs[6]['params']}")


Fitted distribution : gengamma
Parameters          : {'scale': np.float64(76.00128355110742), 'shape1': np.float64(0.6405407480192181), 'shape2': np.float64(120.45170564093303)}
Fit objective (RMSE): 0.1473



ACS fitted for 12 months
January  — model: weibull, params: {'scale': np.float64(0.3511481026983382), 'shape': np.float64(0.7768496260551283)}
July     — model: weibull, params: {'scale': np.float64(0.049999999999999156), 'shape': np.float64(1.6437499999999985)}


In [5]:
# Analyse the observed series — fit seasonal marginals + ACS
print("Fitting stochastic model (analyze_ts) ...")
ts_stats = analyze_ts(Q_obs, dist='gengamma', acs_id='weibull')
print("Done.  Fitted seasonal statistics:")

# report_ts returns a DataFrame with one row per month
stats_df = report_ts(ts_stats)
print(stats_df[['dist', 'scale', 'shape1', 'mean', 'sd', 'p0', 'acs.l.2']].to_string())


Fitting stochastic model (analyze_ts) ...


Done.  Fitted seasonal statistics:
              dist      scale          scale     shape1       mean        sd   p0   acs.l.2
season                                                                                     
month_1   gengamma   1.003543   3.511481e-01   0.976870  11.051547  7.029303  0.0  0.102882
month_2   gengamma  36.173731   7.757137e-01   0.523384  12.616347  7.644519  0.0  0.079441
month_3   gengamma  41.986819   8.224939e-07   0.709123  18.456995  8.361039  0.0  0.081555
month_4   gengamma   1.013950   3.940599e-05   6.047168  28.588973  8.647015  0.0  0.131316
month_5   gengamma   1.019512   1.665548e-61  12.440650  38.615053  8.657339  0.0  0.149832
month_6   gengamma   1.023535   3.587540e-06  30.239539  46.393265  8.201737  0.0  0.014237
month_7   gengamma   1.025129   5.000000e-02  34.661819  49.721187  8.006255  0.0 -0.009968
month_8   gengamma   1.023793   3.697921e-07  27.605471  47.465382  8.523997  0.0  0.031052
month_9   gengamma   1.024521   7.448778e-14 

In [6]:
# Generate N synthetic realisations using generate_ts
# fit_distribution / fit_acs work on raw values, preserving correct scale
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

N_SIM = 20

# Fit the overall (annual) marginal distribution on raw discharge values
dist_fit_full = fit_distribution(Q_obs, dist='gengamma')

# Annual empirical ACS (averaged across months)
acs_fit_full = fit_acs(Q_obs, lag_max=8)
acs_vals = [1.0]
for lag in range(1, 6):
    lag_vals = [m['e_acs'][lag] for m in acs_fit_full if lag < len(m['e_acs'])]
    acs_vals.append(float(np.mean(lag_vals)) if lag_vals else 0.0)

sims = generate_ts(365, dist_fit_full['dist'], dist_fit_full['params_dict'],
                   acs_vals, p0=0.0, n_series=N_SIM)

sim_dates = pd.date_range("1990-01-01", periods=365, freq="D")
fig, ax = plt.subplots(figsize=(14, 5))
for s in sims:
    ax.plot(sim_dates, np.asarray(s), lw=0.5, alpha=0.35, color="steelblue")
ax.plot(Q_obs.loc["1990"].index, Q_obs.loc["1990"].values,
        lw=1.2, color="black", label="Observed 1990")
ax.set_ylabel("Discharge (m³/s)")
ax.set_title(f"generate_ts — {N_SIM} synthetic realisations vs observed", fontsize=13)
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("/tmp/stoch_simulations.png", dpi=100)
plt.close()

sim_arr = np.array([np.asarray(s) for s in sims])
print(f"Observed  — mean: {Q_obs.mean():.2f}, std: {Q_obs.std():.2f}")
print(f"Synthetic — mean: {sim_arr.mean():.2f}, std: {sim_arr.std():.2f}")
print(f"Ensemble shape: {sim_arr.shape} (n_sims × n_days)")
print("Plot saved to /tmp/stoch_simulations.png")


Observed  — mean: 29.97, std: 16.11
Synthetic — mean: 29.80, std: 22.74
Ensemble shape: (20, 365) (n_sims × n_days)
Plot saved to /tmp/stoch_simulations.png


---
## 2. 🌧️ Precipitation — NSRP Model (NEOPRENE)

The **Neyman-Scott Rectangular Pulses Model (NSRPM)** is a process-based stochastic model
specifically designed for precipitation:

- **Physical basis**: storm arrivals (Poisson) → cell clusters (Poisson) → rectangular pulses (exponential duration/intensity)
- **Calibration**: Particle Swarm Optimisation (PSO) matching observed statistics
- **Target statistics**: mean, variance, covariance, probability dry, skewness — by month

### 📐 Model variants

| Class | Model | Sites |
|-------|-------|-------|
| `NSRPModel` | NSRPM | Single-site |
| `STNSRPModel` | ST-NSRPM | Multi-site (spatial) |

> **Requires NEOPRENE:** `pip install NEOPRENE`  
> Developed at IH Cantabria: https://github.com/IHCantabria/NEOPRENE

In [7]:
# Generate a synthetic 30-year daily precipitation series
dates_p   = pd.date_range("1990-01-01", "2019-12-31", freq="D")
prob_wet  = 0.30 + 0.15 * np.sin(2 * np.pi * (np.asarray(dates_p.dayofyear, dtype=float) / 365 - 0.1))
wet       = rng.random(len(dates_p)) < prob_wet
prec      = np.where(wet, rng.exponential(8, len(dates_p)), 0.0)
P_obs     = pd.Series(prec, index=dates_p, name="precipitation")

print(f"Wet-day fraction : {(P_obs > 0).mean():.2f}")
print(f"Mean             : {P_obs.mean():.2f} mm/day")
print(f"Std              : {P_obs.std():.2f} mm/day")
print(f"Max              : {P_obs.max():.1f} mm")

Wet-day fraction : 0.30
Mean             : 2.48 mm/day
Std              : 5.99 mm/day
Max              : 83.9 mm


In [8]:
try:
    # Instantiate the single-site NSRP model
    model = NSRPModel(
        statistics=["mean", "variance", "covariance", "probability_dry"],
        weights=None,   # uniform weights — all statistics equally important
    )
    print(model)
except ImportError as e:
    print(f"NSRPModel requires NEOPRENE: {e}")
    print("  pip install NEOPRENE  # or: pip install git+https://github.com/IHCantabria/NEOPRENE")


In [9]:
try:
    obs_stats = model.compute_statistics(P_obs)
    print("Observed statistics (monthly):")
    print(obs_stats)
except (NameError, ImportError):
    print("(Requires NEOPRENE — model not available)")


(Requires NEOPRENE — model not available)


In [10]:
# Calibrate the NSRP model (requires NEOPRENE — pip install NEOPRENE)
try:
    model.calibrate(P_obs, verbose=False)
    print("Calibration done.")
    print(model.summary())
except (ImportError, RuntimeError) as e:
    print(f"Calibration not available: {e}")
    print("  Install NEOPRENE:  pip install NEOPRENE")


Calibration not available: NEOPRENE is required for stochastic precipitation generation.
Install it with: pip install NEOPRENE
Source: https://github.com/IHCantabria/NEOPRENE
  Install NEOPRENE:  pip install NEOPRENE


In [11]:
# Simulate 100 years of synthetic precipitation (requires NEOPRENE)
try:
    df_sim = model.simulate(year_ini=1990, year_fin=2090)

    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(2, 1, figsize=(14, 7))
    axes[0].bar(P_obs.index, P_obs.values, width=1, color="royalblue", alpha=0.7, label="Observed")
    axes[0].set_ylabel("Precipitation (mm)")
    axes[0].set_title("Observed daily precipitation")
    axes[0].set_xlim(P_obs.index[0], P_obs.index[0] + pd.Timedelta(days=365))
    axes[0].grid(alpha=0.3, axis="y")

    axes[1].bar(df_sim.index, df_sim.values, width=1, color="orange", alpha=0.7, label="Simulated")
    axes[1].set_ylabel("Precipitation (mm)")
    axes[1].set_title("Simulated daily precipitation (1 year sample)")
    axes[1].set_xlim(df_sim.index[0], df_sim.index[0] + pd.Timedelta(days=365))
    axes[1].grid(alpha=0.3, axis="y")

    plt.tight_layout()
    plt.savefig("/tmp/nsrp_simulation.png", dpi=100)
    plt.close()
    print("Simulation plot saved to /tmp/nsrp_simulation.png")
except (ImportError, RuntimeError, AttributeError) as e:
    print(f"Simulation not available: {e}")
    print("  Install NEOPRENE and run calibration first.")


Simulation not available: Call fit() or calibrate() before simulate().
  Install NEOPRENE and run calibration first.


---
## 3. 🗺️ Multi-site precipitation — ST-NSRP Model

The **Spatial-Temporal NSRPM (ST-NSRPM)** extends the single-site model to reproduce
**spatial correlation** between multiple rainfall gauges simultaneously.

In [12]:
try:
    # Multi-site model instantiation
    st_model = STNSRPModel(
        statistics=["mean", "variance", "covariance", "probability_dry"],
    )
    print(st_model)
    print("\nCalibrate with a DataFrame (dates × stations):")
    print("  st_model.fit(df_multisite)")
    print("  df_sim = st_model.simulate(year_ini=1990, year_fin=2090)")
except ImportError as e:
    print(f"NSRPModel requires NEOPRENE: {e}")
    print("  pip install NEOPRENE  # or: pip install git+https://github.com/IHCantabria/NEOPRENE")



Calibrate with a DataFrame (dates × stations):
  st_model.fit(df_multisite)
  df_sim = st_model.simulate(year_ini=1990, year_fin=2090)


---
## 4. 📊 Statistics validation

After generating synthetic series, validate that observed statistics are reproduced.

In [13]:
# Compare observed vs synthetic statistics using generate_ts ensemble
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np, pandas as pd

# Monthly empirical ACS values for each month
monthly_acs = [ts_stats['afits'][m]['e_acs'] for m in range(12)]
monthly_dist_params = [ts_stats['dfits'][m]['params_dict'] for m in range(12)]

# Generate 100 synthetic years (one month at a time, then assemble)
N_YRS = 100
syn_mean_by_month = []
syn_std_by_month  = []

for m in range(12):
    acs_vals_m = [1.0] + monthly_acs[m][1:].tolist()
    sims_m = generate_ts(28, ts_stats['dist'], monthly_dist_params[m],
                         acs_vals_m, p0=0.0, n_series=N_YRS)
    month_arr = np.array([np.asarray(s) for s in sims_m])  # (N_YRS, 28)
    syn_mean_by_month.append(month_arr.mean(axis=1))        # N_YRS means
    syn_std_by_month.append(month_arr.std(axis=1))

obs_monthly_mean = [Q_obs[Q_obs.index.month == m+1].mean() for m in range(12)]
obs_monthly_std  = [Q_obs[Q_obs.index.month == m+1].std()  for m in range(12)]

months = range(1, 13)
labels = ["J","F","M","A","M","J","J","A","S","O","N","D"]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

syn_mean_arr = np.array([np.percentile(syn_mean_by_month[m], [10, 90]) for m in range(12)])
syn_std_arr  = np.array([np.percentile(syn_std_by_month[m],  [10, 90]) for m in range(12)])

ax1.fill_between(list(months), syn_mean_arr[:,0], syn_mean_arr[:,1],
                 alpha=0.35, color="steelblue", label="Synthetic 10-90%")
ax1.plot(list(months), obs_monthly_mean, "ko-", ms=5, label="Observed")
ax1.set_xticks(list(months)); ax1.set_xticklabels(labels)
ax1.set_title("Monthly mean discharge"); ax1.set_ylabel("m³/s")
ax1.legend(); ax1.grid(alpha=0.3)

ax2.fill_between(list(months), syn_std_arr[:,0], syn_std_arr[:,1],
                 alpha=0.35, color="orange", label="Synthetic 10-90%")
ax2.plot(list(months), obs_monthly_std, "ko-", ms=5, label="Observed")
ax2.set_xticks(list(months)); ax2.set_xticklabels(labels)
ax2.set_title("Monthly std discharge"); ax2.set_ylabel("m³/s")
ax2.legend(); ax2.grid(alpha=0.3)

plt.suptitle("Observed vs Synthetic monthly statistics (generate_ts ensemble)", fontsize=12)
plt.tight_layout()
plt.savefig("/tmp/stoch_validation.png", dpi=100)
plt.close()
print("Validation plot saved to /tmp/stoch_validation.png")
print("Mean bias (monthly means): {:.3f}".format(
    abs(np.array(obs_monthly_mean) - np.array([np.mean(syn_mean_by_month[m]) for m in range(12)])).mean()
))


Validation plot saved to /tmp/stoch_validation.png
Mean bias (monthly means): 1.851
